# Project Administrative Boundaries (ArcPy)

**Purpose.** Project African administrative boundaries and calculate area attributes.

**Inputs.** Administrative boundary shapefiles.

**Outputs.** Projected administrative boundary layers.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | ArcPy dependencies
import arcpy
import os
from pathlib import Path

arcpy.env.overwriteOutput = True

In [ ]:
# Cell | Administrative-boundary input and output workspaces
input_folder = r"E:\Africa_Infrastructure_Data\boundaries"
output_folder = r"E:\Africa_Infrastructure_Data\boundaries_project"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)
else:
    pass


In [ ]:
# Cell | Africa Albers Equal-Area coordinate reference system
# EPSG: 102022
# Use Africa Albers Equal Area for country-area calculations.
target_sr = arcpy.SpatialReference(102022)


In [ ]:
# Cell | Feature-class geometry identification
def get_geometry_type(feature_class):
    """Return the ArcPy geometry type of a feature class."""
    desc = arcpy.Describe(feature_class)
    return desc.shapeType

In [ ]:
# Cell | Administrative-area calculation
# Add polygon area in square kilometres after projection.
def add_geometry_field(feature_class, geom_type):
    """Add and calculate length or area for the geometry type."""
    if geom_type == "Polygon":
        field_name = "Area"
        if field_name not in [f.name for f in arcpy.ListFields(feature_class)]:
            arcpy.AddField_management(feature_class, field_name, "DOUBLE")
        arcpy.CalculateField_management(feature_class, field_name, "!shape.area!", "PYTHON3")


In [ ]:
# Cell | Boundary projection and attribute enrichment
def process_feature_class(input_fc, output_fc, target_sr):
    """Project a feature class and calculate geometry attributes."""
    try:
        arcpy.Project_management(input_fc, output_fc, target_sr)
        
        geom_type = get_geometry_type(output_fc)
        
        add_geometry_field(output_fc, geom_type)
        
        return True
    except Exception as e:
        return False

In [ ]:
# Cell | Administrative shapefile discovery
shapefiles = [f for f in os.listdir(input_folder) if f.endswith('.shp')]

print("="*50)
# Project each administrative boundary layer and calculate its geometry attributes.
for shp in shapefiles:
    print(f"  - {shp}")

In [ ]:
# Cell | Batch projection of administrative boundaries
success_count = 0
fail_count = 0

print("="*50)

# Project each administrative boundary layer and calculate its geometry attributes.
for shp in shapefiles:
    input_fc = os.path.join(input_folder, shp)
    output_fc = os.path.join(output_folder, shp)
    
    
    if process_feature_class(input_fc, output_fc, target_sr):
        success_count += 1
    else:
        fail_count += 1